# RuuPhenome — NMR Metabolite Recognition
## Using NMRTransformer (open-source)

**Data sources:**
- Domain 1: Annotated NMR peaks from processed spectrum PDF (plant/serum extracts)
- Domain 2: MTBLS242 — human blood serum, 21 metabolites, ~400 samples

**Pipeline:**
1. Load metabolites + SMILES from Domain 2
2. Predict ¹H chemical shifts via NMRTransformer
3. Match Domain 1 observed peaks to predicted shifts
4. Build abundance matrix for biomarker analysis

In [ ]:
# ── Cell 1: Install NMRTransformer (run once) ──────────────────────────────
# Uncomment and run this cell FIRST if NMRTransformer is not installed

# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'rdkit', 'torch', 'numpy', 'pandas', 'matplotlib', 'seaborn', 'nmrglue'])
# subprocess.run(['git', 'clone', 'https://github.com/liningtonlab/NMRTransformer.git'])
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', 'NMRTransformer/'])
print('Dependencies ready')

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json, os
from pathlib import Path

# Update these paths to your actual file locations
DOMAIN2_NMR_TSV    = '/Users/bigray/Downloads/Domain_2_NMR_results_MTBLS242 (2).tsv'
DOMAIN2_SAMPLE_TSV = '/Users/bigray/Downloads/Domain_2_sample_table_MTBLS242 (2).tsv'
OUTPUT_DIR         = Path('nmr_results')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Paths configured ✓')

In [ ]:
# ── Cell 3: Load Domain 2 Data ─────────────────────────────────────────────
df_raw = pd.read_csv(DOMAIN2_NMR_TSV, sep='\t', low_memory=False)

META_COLS = ['database_identifier', 'chemical_formula', 'smiles', 'inchi',
             'metabolite_identification', 'chemical_shift', 'multiplicity']
SKIP_COLS = set(META_COLS + ['taxid', 'species', 'database', 'database_version',
                               'reliability', 'uri', 'search_engine', 'search_engine_score',
                               'smallmolecule_abundance_sub', 'smallmolecule_abundance_stdev_sub',
                               'smallmolecule_abundance_std_error_sub'])
SAMPLE_COLS = [c for c in df_raw.columns if c not in SKIP_COLS]

df_meta = df_raw[META_COLS].copy()
df_abund = df_raw[SAMPLE_COLS].apply(pd.to_numeric, errors='coerce')

print(f'Metabolites: {len(df_meta)}')
print(f'Samples:     {len(SAMPLE_COLS)}')
print()
df_meta[['metabolite_identification','chemical_formula','smiles']].head(21)

In [ ]:
# ── Cell 4: NMRTransformer — Predict ¹H Chemical Shifts ────────────────────
#
# Try NMRTransformer first; fall back to hardcoded HMDB values if not installed.
#
def predict_with_nmrtransformer(smiles_list):
    try:
        from NMRTransformer.predict import predict_1h_shifts
        results = {}
        for smi in smiles_list:
            try:
                results[smi] = predict_1h_shifts(smi)
                print(f'  ✓ {smi[:40]:40s} → {len(results[smi])} shifts predicted')
            except Exception as e:
                print(f'  ✗ {smi[:40]:40s} → {e}')
                results[smi] = []
        return results, 'NMRTransformer'
    except ImportError:
        return None, None

# Known ¹H shifts from HMDB 5.0 (fallback)
HMDB_KNOWN_SHIFTS = {
    'C[C@@H](O)CC(O)=O':              [1.20, 2.30, 2.44, 4.17],
    'CC(O)=O':                         [1.92],
    'CC(=O)CC(O)=O':                   [2.28, 3.43],
    'C[C@H](N)C(O)=O':                 [1.48, 3.78],
    'CC(C)[C@H](N)C(O)=O':             [0.94, 0.99, 2.28, 3.61],
    'CC[C@H](C)[C@H](N)C(O)=O':        [0.93, 0.95, 1.47, 1.98, 3.68],
    'CC(C)C[C@H](N)C(O)=O':            [0.94, 0.96, 1.70, 1.72, 3.73],
    'N[C@@H](CC(=O)N)C(O)=O':          [2.72, 2.93, 4.01],
    'N[C@@H](CC(O)=O)C(O)=O':          [2.65, 2.84, 3.92],
    'N[C@@H](CCC(=O)N)C(O)=O':         [1.88, 2.13, 2.47, 3.76],
    'N[C@@H](CCC(O)=O)C(O)=O':         [1.88, 2.12, 2.35, 3.76],
    'N[C@@H](Cc1ccc(O)cc1)C(O)=O':     [3.04, 3.19, 6.89, 7.18],
    'N[C@@H](Cc1cnc[nH]1)C(O)=O':      [3.19, 3.28, 7.08, 7.78],
    'OC(=O)CCC(O)=O':                  [2.41],
    'OC(CC(O)=O)C(O)=O':               [2.54, 2.67],
    'OC(=O)CC(O)C(O)=O':              [2.65, 4.29],
    '[C@@H]1([C@H]([C@@H]([C@H](C(O1)O)O)O)O)O': [3.25, 3.39, 3.53, 3.84, 5.22],
    'OC[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O': [3.25, 3.52, 3.71, 3.84, 5.22],
    'OCC(O)(CO)CO':                    [3.56, 3.65],
    'OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O': [3.40, 3.74, 4.04, 5.17],
    'C([C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)O)O)O)O)O': [3.35, 3.52, 3.56, 3.83, 5.09],
}

smiles_list = df_meta['smiles'].dropna().tolist()
predicted_shifts, method = predict_with_nmrtransformer(smiles_list)

if predicted_shifts is None:
    print('[INFO] NMRTransformer not installed — using HMDB fallback shifts')
    predicted_shifts = {smi: HMDB_KNOWN_SHIFTS.get(smi, []) for smi in smiles_list}
    method = 'HMDB fallback'

print(f'\nMethod: {method}')
print(f'Predictions for {sum(1 for v in predicted_shifts.values() if v)} / {len(smiles_list)} SMILES')

# Attach predictions back to metadata
df_meta['predicted_1h_shifts'] = df_meta['smiles'].map(
    lambda s: json.dumps(predicted_shifts.get(s, [])) if s else '[]'
)
df_meta[['metabolite_identification','predicted_1h_shifts']].head(10)

In [ ]:
# ── Cell 5: Domain 1 — Observed NMR Peaks ─────────────────────────────────
# Extracted from Domain_1_processed_NMR_spectrum.pdf annotation
domain1_peaks = pd.DataFrame([
    {'metabolite': 'Asparagine',      'observed_shift': 2.87, 'region': 'aliphatic'},
    {'metabolite': 'Aspartate',       'observed_shift': 2.65, 'region': 'aliphatic'},
    {'metabolite': '4-Aminobutyrate', 'observed_shift': 1.89, 'region': 'aliphatic'},
    {'metabolite': 'Citrate',         'observed_shift': 2.54, 'region': 'aliphatic'},
    {'metabolite': 'Succinate',       'observed_shift': 2.41, 'region': 'aliphatic'},
    {'metabolite': 'Glutamate',       'observed_shift': 2.35, 'region': 'aliphatic'},
    {'metabolite': 'Acetate',         'observed_shift': 1.92, 'region': 'aliphatic'},
    {'metabolite': 'Alanine',         'observed_shift': 1.48, 'region': 'aliphatic'},
    {'metabolite': 'Threonine',       'observed_shift': 1.33, 'region': 'aliphatic'},
    {'metabolite': 'Valine',          'observed_shift': 0.98, 'region': 'aliphatic'},
    {'metabolite': 'Leucine',         'observed_shift': 0.96, 'region': 'aliphatic'},
    {'metabolite': 'Isoleucine',      'observed_shift': 0.93, 'region': 'aliphatic'},
    {'metabolite': 'Choline',         'observed_shift': 3.21, 'region': 'oxygenated'},
    {'metabolite': 'Meso-Inositol',   'observed_shift': 3.27, 'region': 'oxygenated'},
    {'metabolite': 'Methanol',        'observed_shift': 3.36, 'region': 'oxygenated'},
    {'metabolite': 'Sucrose',         'observed_shift': 5.40, 'region': 'anomeric'},
    {'metabolite': 'Glucose',         'observed_shift': 5.22, 'region': 'anomeric'},
    {'metabolite': 'Xylose',          'observed_shift': 5.17, 'region': 'anomeric'},
    {'metabolite': 'Cytidine',        'observed_shift': 5.90, 'region': 'anomeric'},
    {'metabolite': 'Uridine',         'observed_shift': 5.89, 'region': 'anomeric'},
    {'metabolite': 'Tyrosine',        'observed_shift': 6.89, 'region': 'aromatic'},
    {'metabolite': 'Histidine',       'observed_shift': 7.08, 'region': 'aromatic'},
    {'metabolite': 'Tryptophan',      'observed_shift': 7.33, 'region': 'aromatic'},
    {'metabolite': 'Phenylalanine',   'observed_shift': 7.32, 'region': 'aromatic'},
    {'metabolite': 'Guanine',         'observed_shift': 7.76, 'region': 'aromatic'},
    {'metabolite': 'Xanthurinate',    'observed_shift': 7.82, 'region': 'aromatic'},
    {'metabolite': 'Nicotinate',      'observed_shift': 8.62, 'region': 'aromatic'},
    {'metabolite': 'Formate',         'observed_shift': 8.45, 'region': 'aromatic'},
    {'metabolite': 'Adenosine',       'observed_shift': 8.34, 'region': 'aromatic'},
    {'metabolite': 'Malate',          'observed_shift': 2.65, 'region': 'aliphatic'},
])

# Simulate a spectrum plot
fig, ax = plt.subplots(figsize=(14, 4))
region_colors = {'aliphatic': '#2196F3', 'oxygenated': '#4CAF50',
                 'anomeric': '#FF9800', 'aromatic': '#9C27B0'}
for _, row in domain1_peaks.iterrows():
    c = region_colors[row['region']]
    ax.axvline(row['observed_shift'], color=c, alpha=0.7, lw=1.5)
    ax.text(row['observed_shift'], 0.8 + np.random.uniform(-0.1, 0.1),
            row['metabolite'], fontsize=6, rotation=90, ha='center', color=c)

ax.set_xlim(10, 0)  # NMR x-axis is reversed
ax.set_xlabel('Chemical Shift (ppm)', fontsize=12)
ax.set_ylabel('Relative Intensity', fontsize=12)
ax.set_title('Domain 1 — Annotated ¹H NMR Spectrum', fontsize=14)
patches = [mpatches.Patch(color=c, label=r) for r, c in region_colors.items()]
ax.legend(handles=patches, loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'domain1_spectrum.png', dpi=150)
plt.show()
print(f'Domain 1: {len(domain1_peaks)} annotated peaks')

In [ ]:
# ── Cell 6: Match Domain 2 candidates vs Domain 1 peaks ───────────────────
TOLERANCE = 0.05  # ppm

records = []
for _, row in df_meta.iterrows():
    smi = row['smiles'] if pd.notna(row['smiles']) else ''
    pred = predicted_shifts.get(smi, [])
    name = row['metabolite_identification']

    matched, matched_info = 0, []
    for _, obs in domain1_peaks.iterrows():
        for p in pred:
            if abs(p - obs['observed_shift']) <= TOLERANCE:
                matched += 1
                matched_info.append(f"{obs['metabolite']}≈{p:.2f}ppm")
                break

    score = (matched / max(len(pred), 1)) * 100 if pred else 0
    records.append({
        'metabolite':       name,
        'chebi':            row['database_identifier'],
        'formula':          row['chemical_formula'],
        'predicted_shifts': pred,
        'n_shifts_pred':    len(pred),
        'peaks_matched':    matched,
        'match_score_%':    round(score, 1),
        'matched_detail':   '; '.join(matched_info),
    })

df_match = pd.DataFrame(records).sort_values('match_score_%', ascending=False)
df_match.to_csv(OUTPUT_DIR / 'peak_match_scores.csv', index=False)

print('Top metabolite matches (Domain 2 vs Domain 1 peaks):')
df_match[['metabolite','chebi','n_shifts_pred','peaks_matched','match_score_%','matched_detail']].head(10)

In [ ]:
# ── Cell 7: Abundance Heatmap (Metabolite × Sample) ───────────────────────
# Normalize each metabolite (z-score across samples)
df_z = df_abund.apply(lambda col: (col - col.mean()) / col.std(), axis=1)
df_z.index = df_meta['metabolite_identification'].values

# Take first 40 samples for visualization
df_z_plot = df_z.iloc[:, :40].dropna(how='all')

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(df_z_plot, cmap='RdBu_r', center=0, ax=ax,
            yticklabels=True, xticklabels=False,
            cbar_kws={'label': 'Z-score abundance'})
ax.set_title('Metabolite Abundance Heatmap — MTBLS242 (first 40 samples)', fontsize=13)
ax.set_xlabel('Samples (preop)', fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'abundance_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ── Cell 8: CV% Filter — Stable Biomarker Candidates ─────────────────────
# Low CV% = metabolite is consistently detected across samples = good biomarker
df_meta['mean_abund']   = df_abund.mean(axis=1).values
df_meta['stdev_abund']  = df_abund.std(axis=1).values
df_meta['cv_pct']       = (df_meta['stdev_abund'] / df_meta['mean_abund'] * 100).abs()
df_meta['detected_pct'] = (df_abund.notna().sum(axis=1) / len(SAMPLE_COLS) * 100).values

df_stable = df_meta.sort_values('cv_pct')
print('Most stable metabolites (low CV = consistent across samples):')
df_stable[['metabolite_identification','mean_abund','cv_pct','detected_pct']].head(10)

In [ ]:
# ── Cell 9: Export complete annotated table ────────────────────────────────
df_export = df_meta.copy()
df_export['domain1_match_score_%'] = df_match.set_index('metabolite').reindex(
    df_export['metabolite_identification'])['match_score_%'].values

export_path = OUTPUT_DIR / 'annotated_metabolites.csv'
df_export.to_csv(export_path, index=False)
print(f'Saved: {export_path}')
df_export[['metabolite_identification','chemical_formula','predicted_1h_shifts',
            'mean_abund','cv_pct','domain1_match_score_%']]

## Next Steps

| Step | What to do |
|------|------------|
| **Install NMRTransformer** | Run `bash setup.sh` in the `nmr_pipeline/` folder |
| **Run with real NMR files** | Load Bruker `.fid` or Varian files with `nmrglue` |
| **Add sample metadata** | Join sample table (time point: preop/postop) for comparison |
| **Biomarker analysis** | Use stable metabolites (low CV%) for diabetes/hypertension classification |
| **Integrate into RuuPhenome** | Expose this pipeline as a REST API endpoint (`/api/nmr/analyze`) |